# Un réseau de neurones NumPy : voir la rétropropagation

**Objectifs observables (90–120 min).** Suivre les formes d'une forward pass, calculer la binary cross-entropy, vérifier numériquement un gradient, entraîner deux couches et visualiser leur frontière de décision.

**Prérequis :** produit matriciel, dérivée, règle de chaîne et classification. XOR est un problème synthétique de quatre points : une frontière droite ne peut pas le résoudre. Ce notebook enseigne le mécanisme, sans prétendre mesurer la généralisation.

## 1. Des dimensions aux calculs

Avec `X` de forme `(n, 2)` et `W1` de forme `(2, h)`, $Z_1=XW_1+b_1$ est `(n,h)`. Le biais `b1` de forme `(1,h)` est diffusé sur les `n` lignes. $A_1=\tanh(Z_1)$ puis $Z_2=A_1W_2+b_2$ de forme `(n,1)`. $P=\sigma(Z_2)$ donne une probabilité par point.

Deux couches permettent une transformation intermédiaire de l'espace. Leur nombre de poids, pour `h=4`, est $2\times4+4+4\times1+1=17$. Comptez-les dans le code et comparez au nombre de données : quatre points seulement.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([[0], [1], [1], [0]], dtype=float)
rng = np.random.default_rng(3)
hidden_units = 4
W1 = rng.normal(0, 0.6, size=(2, hidden_units)); b1 = np.zeros((1, hidden_units))
W2 = rng.normal(0, 0.6, size=(hidden_units, 1)); b2 = np.zeros((1, 1))
print('X:', X.shape, 'W1:', W1.shape, 'W2:', W2.shape)

## 2. Forward pass et perte

Pour la cible $y_i\in\{0,1\}$, la binary cross-entropy est $-\frac1n\sum_i [y_i\log p_i+(1-y_i)\log(1-p_i)]$. Une prédiction confiante et fausse coûte beaucoup. Nous limitons `p` uniquement avant le logarithme afin d'éviter `log(0)` en virgule flottante.

**Petit calcul.** Si `y=1` et `p=0,8`, la contribution est $-\log(0,8)\approx0,223$ ; si `p=0,1`, elle est $-\log(0,1)\approx2,303$.

In [ ]:
def sigmoid(value):
    return 1 / (1 + np.exp(-value))

def forward(features, w1, b1, w2, b2):
    z1 = features @ w1 + b1
    a1 = np.tanh(z1)
    logits = a1 @ w2 + b2
    probabilities = sigmoid(logits)
    return z1, a1, logits, probabilities

def binary_cross_entropy(target, probability):
    clipped = np.clip(probability, 1e-8, 1 - 1e-8)
    return -np.mean(target * np.log(clipped) + (1 - target) * np.log(1 - clipped))

_, hidden, logits, probability = forward(X, W1, b1, W2, b2)
print('hidden/output:', hidden.shape, probability.shape)
print('initial BCE:', round(float(binary_cross_entropy(y, probability)), 4))

In [ ]:
print('BCE if true label = 1 and p = 0.8:', round(float(-np.log(0.8)), 3))
print('BCE if true label = 1 and p = 0.1:', round(float(-np.log(0.1)), 3))
_, a1_check, _, p_check = forward(X, W1, b1, W2, b2)
analytic_w2_00 = (a1_check.T @ ((p_check - y) / len(X)))[0, 0]
epsilon = 1e-5
w2_plus = W2.copy(); w2_plus[0, 0] += epsilon
w2_minus = W2.copy(); w2_minus[0, 0] -= epsilon
loss_plus = binary_cross_entropy(y, forward(X, W1, b1, w2_plus, b2)[-1])
loss_minus = binary_cross_entropy(y, forward(X, W1, b1, w2_minus, b2)[-1])
finite_w2_00 = (loss_plus - loss_minus) / (2 * epsilon)
print('Gradient W2[0,0] analytic / finite:', round(float(analytic_w2_00), 6), round(float(finite_w2_00), 6))
print('Absolute difference:', abs(analytic_w2_00 - finite_w2_00))

## 3. Rétropropagation et vérification

Avec sigmoid puis binary cross-entropy, $\partial L/\partial z_2=(p-y)/n$. On a $\partial L/\partial W_2=A_1^T\delta_2$, puis $\delta_1=(\delta_2W_2^T)\odot(1-A_1^2)$ et $\partial L/\partial W_1=X^T\delta_1$. Les formes sont respectivement `(h,1)`, `(n,h)` et `(2,h)`.

Avant d'entraîner, vérifions une composante de gradient avec les différences finies $[L(w+\epsilon)-L(w-\epsilon)]/(2\epsilon)$. Un fort désaccord signale une erreur de dérivée, de facteur $1/n$ ou de forme.

In [ ]:
learning_rate = 0.7
loss_history = []
for epoch in range(5_000):
    z1, a1, _, p = forward(X, W1, b1, W2, b2)
    loss_history.append(float(binary_cross_entropy(y, p)))
    delta2 = (p - y) / len(X)
    grad_w2 = a1.T @ delta2
    grad_b2 = delta2.sum(axis=0, keepdims=True)
    delta1 = (delta2 @ W2.T) * (1 - a1 ** 2)
    grad_w1 = X.T @ delta1
    grad_b1 = delta1.sum(axis=0, keepdims=True)
    W2 -= learning_rate * grad_w2; b2 -= learning_rate * grad_b2
    W1 -= learning_rate * grad_w1; b1 -= learning_rate * grad_b1
print('final BCE:', round(loss_history[-1], 4))
print('probabilities:', forward(X, W1, b1, W2, b2)[-1].round(3).ravel())

## 4. Visualiser la loss **et** la frontière

La courbe de loss montre l'optimisation des quatre points. La carte de probabilité révèle ce que le réseau fait entre eux : les couleurs correspondent à $P(y=1|x)$ et le contour blanc à un seuil de 0,5. La prédiction entre les points n'est pas validée par les données.

In [ ]:
plt.plot(loss_history)
plt.xlabel('Epoch'); plt.ylabel('Binary cross-entropy')
plt.title('Loss du MLP NumPy'); plt.grid(alpha=0.2); plt.show()
print('prédictions binaires:', (forward(X, W1, b1, W2, b2)[-1] >= 0.5).astype(int).ravel())

In [ ]:
axis = np.linspace(-0.35, 1.35, 170)
mesh_x, mesh_y = np.meshgrid(axis, axis)
mesh_points = np.column_stack([mesh_x.ravel(), mesh_y.ravel()])
mesh_probability = forward(mesh_points, W1, b1, W2, b2)[-1].reshape(mesh_x.shape)
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.contourf(mesh_x, mesh_y, mesh_probability, levels=np.linspace(0, 1, 21), cmap='RdBu_r', vmin=0, vmax=1)
ax.contour(mesh_x, mesh_y, mesh_probability, levels=[0.5], colors='white', linewidths=2)
ax.scatter(X[:, 0], X[:, 1], c=y.ravel(), cmap='RdBu_r', vmin=0, vmax=1, edgecolors='black', s=125)
for point, label in zip(X, y.ravel()):
    ax.annotate(str(int(label)), point + [0.04, 0.04])
ax.set(xlabel='Entrée x₁', ylabel='Entrée x₂', title='Probabilité apprise sur XOR (contour = 0,5)')
fig.colorbar(image, ax=ax, label='P(classe 1)')
plt.show()

## 5. Relation avec PyTorch et limites

Autograd calcule la même règle de chaîne à partir des opérations exécutées. Un framework gère aussi les tensors, optimizers, mini-batchs et devices. **Dropout** coupe aléatoirement des activations pendant l'entraînement pour réduire certaines coadaptations ; **batch normalization** normalise des activations avec des statistiques de lot et un état distinct en évaluation. Aucun de ces deux mécanismes n'est nécessaire pour ce XOR.

Les surfaces de loss non convexes, l'initialisation et le learning rate influencent l'issue. Les quatre points ne démontrent ni robustesse au bruit ni fiabilité hors du domaine observé.

## Exercices

1. **Easy :** retrouvez les 17 paramètres et les formes de tous les intermédiaires.
2. **Medium :** vérifiez par différences finies une composante de `W1` et de `b2`.
3. **Hard :** entraînez 5 initialisations avec 2 puis 8 unités cachées ; comparez la loss finale et la frontière. Ajoutez du bruit seulement si vous définissez un split d'évaluation. Expliquez ce que quatre points ne permettent pas d'estimer.